# 🤖 Kronos Trading — Backtest trên Google Colab

Notebook chạy **backtest có tín hiệu Kronos + SL/TP + sizing theo R + chi phí thật** trên dữ liệu MetaTrader 5 của bạn (BTCUSDT, EURUSD, XAUUSD...).

**Cách dùng:** Runtime → Change runtime type → chọn **GPU (T4)**, rồi chạy lần lượt từng cell từ trên xuống.

> ⚠️ Đây là công cụ nghiên cứu, KHÔNG phải lời khuyên đầu tư. Backtest quá khứ không đảm bảo tương lai. Luôn paper-trade trước.

## Bước 1 — Kiểm tra GPU

In [ ]:
!nvidia-smi -L || echo 'Chưa bật GPU: Runtime > Change runtime type > GPU'

## Bước 2 — Tải code & cài đặt thư viện

Clone repo (nhánh chứa package `trading`) và cài dependency.

In [ ]:
REPO_URL = 'https://github.com/nguyenlam19122/Kronoss.git'
BRANCH   = 'claude/affectionate-fermi-b956vh'   # đổi thành 'main' sau khi bạn merge

import os
if not os.path.isdir('Kronoss'):
    !git clone --quiet --branch $BRANCH $REPO_URL
%cd Kronoss
!pip install -q -r requirements.txt -r trading/requirements.txt
print('✅ Sẵn sàng')

## Bước 3 — Tải file dữ liệu MT5 của bạn lên

Chạy cell dưới rồi chọn 1 hoặc nhiều file CSV xuất từ MetaTrader (định dạng `<DATE> <TIME> <OPEN>...<SPREAD>`).

In [ ]:
from google.colab import files
uploaded = files.upload()
os.makedirs('trading/mydata', exist_ok=True)
for name, content in uploaded.items():
    with open(f'trading/mydata/{name}', 'wb') as f:
        f.write(content)
print('Đã lưu:', os.listdir('trading/mydata'))

## Bước 4 — Cấu hình backtest

**Chỉ cần chỉnh ở cell này.** Quan trọng nhất: `DATA_FILE`, `RISK_AMOUNT` (1R), `SL_ATR`, `RR`, và chi phí.

In [ ]:
# ===== NGUỒN DỮ LIỆU =====
DATA_FILE   = 'trading/mydata/XAUUSDm_M5.csv'  # << đổi sang tên file bạn vừa upload
USE_LAST_N  = 8000      # chỉ test N nến gần nhất cho nhanh; đặt None để chạy toàn bộ

# ===== MODEL =====
MODEL_NAME    = 'NeoQuasar/Kronos-small'   # -small (nhanh) | -base (chất lượng hơn, chậm)
LOOKBACK      = 256     # số nến lịch sử nạp vào model (<= 512)
PRED_LEN      = 24      # số nến dự báo mỗi lần (24 nến M5 = 2 giờ)
SIGNAL_EVERY  = 12      # chạy model mỗi 12 nến (1 giờ). Tăng lên để chạy nhanh hơn
SAMPLE_COUNT  = 1       # 2-3 = trung bình nhiều đường dự báo, ổn định hơn nhưng chậm hơn

# ===== TÍN HIỆU =====
SIGNAL_MODE      = 'mean'   # 'mean' | 'endpoint' | 'slope'
LONG_THRESHOLD   = 0.0      # vd 0.001 = chỉ Long khi dự báo tăng > 0.1%
SHORT_THRESHOLD  = 0.0
ALLOW_SHORT      = True

# ===== STOP-LOSS / TAKE-PROFIT =====
SL_ATR     = 1.5    # stop = 1.5 * ATR = 1R
RR         = 2.0    # take-profit = 2R (R:R = 2:1)
ATR_PERIOD = 14
MAX_HOLD   = None   # None = giữ tối đa PRED_LEN nến rồi thoát

# ===== TRAILING STOP (tùy chọn) =====
TRAIL            = False  # True = bật trailing stop (dời SL theo giá để khóa lời)
TRAIL_ATR        = 1.5    # trailing cách đỉnh/đáy thuận lợi bao nhiêu ATR
TRAIL_ACTIVATE_R = 1.0    # chỉ bắt đầu trail sau khi lời +1R
# Mẹo: để 'thả lệnh thắng chạy', bật TRAIL=True và đặt RR=0 (tắt TP cố định)

# ===== VỐN & RỦI RO (1R = 25 đô) =====
INITIAL_CAPITAL = 5000.0
RISK_MODE     = 'fixed'   # 'fixed' = 1R cố định $ | 'percent' = 1R theo %% vốn
RISK_AMOUNT   = 25.0      # 1R = $25
RISK_PCT      = 0.005     # dùng khi RISK_MODE='percent' (0.5%% vốn)
MAX_LEVERAGE  = 30        # giới hạn đòn bẩy (None = không giới hạn). Lưu ý sizing theo R có thể tạo đòn bẩy cao!

# ===== CHI PHÍ =====
USE_DATA_SPREAD      = True   # dùng spread THẬT trong file MT5 của bạn
SLIPPAGE_POINTS      = 5      # trượt giá mỗi chiều (points)
COMMISSION_BPS       = 0.0    # hoa hồng mỗi chiều (basis points của notional)
COMMISSION_PER_TRADE = 0.0    # hoa hồng cố định mỗi lệnh ($)
print('Đã cấu hình.')

## Bước 5 — Nạp dữ liệu

In [ ]:
from trading.data import load_csv
df = load_csv(DATA_FILE)            # tự nhận diện định dạng MT5
if USE_LAST_N:
    df = df.iloc[-USE_LAST_N:]
print(f'{len(df)} nến | {df.index.min()} -> {df.index.max()}')
print(f"point = {df.attrs.get('point')} | spread theo nến: {'CÓ' if 'spread' in df.columns else 'KHÔNG'}")
df.tail(3)

## Bước 6 — Nạp model Kronos

Lần đầu sẽ tải weights từ Hugging Face (vài chục giây).

In [ ]:
from trading.predictor import load_kronos_predictor, make_kronos_predict_fn
predictor  = load_kronos_predictor(MODEL_NAME, device='cuda:0', max_context=512)
predict_fn = make_kronos_predict_fn(predictor, sample_count=SAMPLE_COUNT)
print('✅ Model sẵn sàng')

## Bước 7 — Chạy backtest

In [ ]:
from trading.trade_sim import TradeConfig, run_trade_sim, save_trade_results, format_trade_metrics
from trading.signals import SignalConfig
from trading.sizing import SizingConfig

cfg = TradeConfig(
    lookback=LOOKBACK, pred_len=PRED_LEN, signal_every=SIGNAL_EVERY,
    signal=SignalConfig(mode=SIGNAL_MODE, long_threshold=LONG_THRESHOLD,
                        short_threshold=SHORT_THRESHOLD, allow_short=ALLOW_SHORT),
    atr_period=ATR_PERIOD, sl_atr=SL_ATR, rr=RR, max_hold=MAX_HOLD,
    trail=TRAIL, trail_atr=TRAIL_ATR, trail_activate_r=TRAIL_ACTIVATE_R,
    sizing=SizingConfig(mode=RISK_MODE, risk_amount=RISK_AMOUNT,
                        risk_pct=RISK_PCT, max_leverage=MAX_LEVERAGE),
    initial_capital=INITIAL_CAPITAL,
    use_data_spread=USE_DATA_SPREAD, slippage_points=SLIPPAGE_POINTS,
    commission_per_notional=COMMISSION_BPS/1e4, commission_per_trade=COMMISSION_PER_TRADE,
)
result = run_trade_sim(df, predict_fn, cfg, verbose=False)
print(format_trade_metrics(result['metrics']))

## Bước 8 — Biểu đồ & bảng lệnh

In [ ]:
name  = df.attrs.get('instrument', 'run')
paths = save_trade_results(result, 'trading/results', name)
from IPython.display import Image, display
display(Image(paths['chart']))
result['trades'].tail(10)

## Bước 9 — Tinh chỉnh & lưu ý quan trọng

- **Đòn bẩy:** sizing theo R với stop ATR hẹp có thể tạo `Avg leverage` rất cao. Dùng `MAX_LEVERAGE` để chặn.
- **Chi phí:** xem dòng `spread / slippage / commission` — trên khung M5 giao dịch dày, chi phí có thể ăn hết lợi nhuận. Tăng `SIGNAL_EVERY`, `LONG_THRESHOLD`, hoặc dùng khung lớn hơn để giảm số lệnh.
- **Trailing stop:** bật `TRAIL=True` để khóa lời. Thử cả `RR=0` (bỏ TP, thả lệnh thắng chạy) — trong test của mình kiểu này cải thiện rõ trên Vàng.
- **Tín hiệu yếu?** Thử `SAMPLE_COUNT=3`, đổi `SIGNAL_MODE`, hoặc nâng ngưỡng vào lệnh.
- **Overfitting:** đừng chỉnh tham số đến khi đẹp trên 1 đoạn dữ liệu. Hãy kiểm tra trên nhiều giai đoạn / nhiều sản phẩm (chạy lại với file BTC, EUR).
- **So sánh baseline:** đặt `MODEL_NAME` không đổi nhưng thử `from trading.baselines import momentum_predict_fn` để xem Kronos có thắng baseline ngây thơ không.

Chạy lại từ Bước 4 với tham số khác để so sánh.